In [18]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

# Ativa o tracing do LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY") or getpass.getpass("LangSmith API Key: ")
os.environ["LANGSMITH_PROJECT"] = "rag-curso-alura"  # ou o nome do projeto que preferir


In [19]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

pdfs = DirectoryLoader(r"G:\Meu Drive\01-Alura\AIEngineering\05 - LangChain Técnicas Avançadas de RAG\4910-LangChain-Tecnicas-Avancas-de-RAG\documentos", glob="*.pdf", loader_cls=PyPDFLoader).load()


In [20]:
len(pdfs)

59

In [21]:
from transformers import AutoTokenizer

In [22]:
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

In [23]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer, chunk_size=1250, chunk_overlap=150
)

In [24]:
pedacos = splitter.split_documents(pdfs)

In [25]:
len(pedacos)

59

In [26]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="bge-m3:567m")

vector_store = FAISS.from_documents(
    documents=pedacos, embedding=embeddings
)

In [27]:
retriever = vector_store.as_retriever()

In [28]:
from langchain_ollama.llms import OllamaLLM

modelo = OllamaLLM(model="gemma3:4b")

In [29]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [30]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda sempre em português brasileiro usando exclusivamente o conteúdo fornecido.\n\nContexto:\n{contexto}"),
        ("human", "{query}")
    ]
)


In [31]:
from langchain_core.output_parsers import StrOutputParser

cadeia = prompt | modelo | StrOutputParser()

In [32]:
pergunta = "Como fazer um seguro viagem?"

trechos = retriever.invoke(pergunta)
for i, trecho in enumerate(trechos, 1):
    origem = trecho.metadata.get("source", "Desconhecido")
    pagina = trecho.metadata.get("page", 0)
    print(f"--- Trecho {i} | Arquivo: {origem} | Página: {pagina} ---")
    print(trecho.page_content[:300]) # Primeiros 300 caracteres
    print("\n")

contexto = "\n\n".join(trecho.page_content for trecho in trechos)

cadeia.invoke({"query": pergunta, "contexto": contexto})


--- Trecho 1 | Arquivo: G:\Meu Drive\01-Alura\AIEngineering\05 - LangChain Técnicas Avançadas de RAG\4910-LangChain-Tecnicas-Avancas-de-RAG\documentos\GTB_platinum_Nov23.pdf | Página: 21 ---
Versão: novembro/2021  
feita se não é superior ao custo médio de tais serviços e fornecimentos na localidade onde  receberam, 
considerando a natureza e a gravidade da Doença Súbita ou Acidente no relação com os quais esses 
serviços e fornecimentos são recebidos. 
Viagem Segurada: É o período de t


--- Trecho 2 | Arquivo: G:\Meu Drive\01-Alura\AIEngineering\05 - LangChain Técnicas Avançadas de RAG\4910-LangChain-Tecnicas-Avancas-de-RAG\documentos\GTB_platinum_Nov23.pdf | Página: 13 ---
Versão: novembro/2021  
Aspectos Importantes: 
- 
- As viagens estão cobertas por um período máximo de 31 (Trinta e um) dias consecutivos a partir 
da primeira data de embarque de cada viagem. 
- As Despesas Médicas estão cobertas até o valor máximo de benefício de USD† 25.000 por Pessoa 
Elegível. 


--- Trecho 3

'Para fazer um seguro viagem, siga os passos abaixo, com base nas informações fornecidas:\n\n1.  **Verifique sua elegibilidade:** Você precisa ser portador do cartão Mastercard Platinum™ e seus dependentes (cônjuges, companheiros e filhos dependentes) viajando juntos ou separados.\n\n2.  **Emita o Bilhete de Seguro:** Acesse o portal www.aig.com/Mastercard/pt e emita o seu Bilhete de Seguro Viagem. Este documento é essencial para ter cobertura. O bilhete tem vigência de 12 meses a partir da data de emissão, e somente viagens iniciadas dentro deste período serão cobertas. É imprescindível apresentar o Bilhete de Seguro no caso de eventual ocorrência/sinistro.\n\n3.  **Condições para cobertura:** Para que você seja elegível à cobertura, o custo total da passagem de um Transporte Público Autorizado deve ser pago com o seu cartão Mastercard Platinum TM ou com pontos ganhos em um Programa de Recompensas associado ao cartão.\n\n4.  **Cobertura e Benefícios:** O seguro oferece cobertura para 

In [33]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "contexto": retriever | format_docs,
        "query": RunnablePassthrough()
    }
    | prompt
    | modelo
    | StrOutputParser()
)


In [34]:
rag_chain.invoke(pergunta)

'Para fazer um seguro viagem, siga estas etapas, considerando as informações fornecidas:\n\n1.  **Verifique a Elegibilidade:** Certifique-se de que você se qualifica para o MasterAssist Plus, que está disponível para portadores de cartões Mastercard Platinum™ e seus dependentes.\n\n2.  **Emita o Bilhete de Seguro:** Acesse o portal www.aig.com/Mastercard/pt e emita o seu Bilhete de Seguro Viagem. Este documento é essencial para acionar a cobertura. O bilhete tem validade de 12 meses a partir da data de emissão e somente cobre viagens iniciadas após essa data.\n\n3.  **Viaje com a Passagem Corretamente Registrada:** Para que a viagem seja coberta, você deve pagar a passagem para transporte público autorizado com seu cartão Mastercard Platinum™ ou com pontos ganhos através de um programa de recompensas associado a ele. Inclua todos os impostos, custos de envio e manuseio.\n\n4.  **Entenda as Coberturas:** O seguro viagem cobre despesas médicas e hospitalares (até USD 25.000), traslado mé

In [35]:
query_model = OllamaLLM(model="gemma3:1b")

In [36]:
rewriter_prompt_template = """
Gere consulta de pesquisa para o banco de dados de vetores (Vector DB) a partir de uma pergunta do usuário,
permitindo uma resposta mais precisa por meio da busca semantica.
Basta retornar a consulta revisada do Vector DB, entre aspas.

Pergunta do usuário: {user_question}

Consulta revisada do Vector DB:
"""

In [45]:
from langchain_core.prompts import PromptTemplate

# Transforma a string em um PromptTemplate do LangChain
rewriter_prompt = PromptTemplate.from_template(rewriter_prompt_template)

# Agora sim você pode encadear com o pipe (|)
rewriter_chain = rewriter_prompt | query_model | StrOutputParser()


In [53]:
print(pergunta)
rewriter_chain.invoke({"user_question": pergunta})

Como fazer um seguro viagem?


'Seguro viagem\n'

In [54]:
rewriter_rag_chain = (
    {
        "contexto": RunnablePassthrough() | rewriter_chain | retriever | format_docs,
        "query": RunnablePassthrough()
    }
    | prompt
    | modelo
    | StrOutputParser()
)

In [56]:
rewriter_rag_chain.invoke(pergunta)

'Para fazer um seguro viagem, siga estas etapas com base nas informações fornecidas:\n\n1.  **Verifique os Requisitos:** Certifique-se de que a viagem se enquadra nas coberturas oferecidas. A cobertura máxima de USD 25.000 para Despesas Médicas e Hospitalares é válida para viagens de até 31 dias consecutivos a partir da primeira data de embarque.\n\n2.  **Entenda as Coberturas:** As coberturas incluem:\n    *   **Despesas Médicas e Hospitalares:** Até USD 25.000 por pessoa elegível.\n    *   **Traslado Médico (Remoção Médica):** Até USD 50.000.\n    *   **Prorrogação de Estadia:** Até USD 150 por dia por até 5 dias.\n    *   **Acompanhante em caso de hospitalização prolongada:** Passagem de ida e volta, USD 150 por dia por até 5 dias.\n    *   **Retorno de Menores/Idosos:** Até USD 10.000.\n    *   **Traslado de Corpo (Repatriação Funerária):** Até USD 25.000.\n    *   **Regresso Sanitário (Repatriação Médica):** Até USD 50.000.\n\n3.  **Serviços Sem Desembolso de Dinheiro:** O MasterA